# 27 · Revision-4 training runs — session B (T4 GPU; run sessions A and B at the same time)
Two copies of this notebook (A and B) share the work through lock files; every run is trained once. Resumable: after a disconnect,
**Run all** again. 60 runs of the trigonometric control (g_tr, p_tr; seeds 0–2; folds 0–4; both learning-rate schedules), about
13 min each on a T4 (≈ 6–7 h per session with two sessions). Settings exactly as notebook 13 (batch 16, Adam 1e-4, up to 200 epochs,
early stopping on the validation loss after epoch 15 with patience 10, best checkpoint on the validation loss, learning rate halved
after four epochs without improvement of the training loss or of the validation loss, respectively). The prespecified text (cell 1) is stored with a time stamp before the first run.

In [ ]:
SESSION = 'B'
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, time, gc, hashlib, random
import numpy as np, pandas as pd
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
FROZ_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit_frozen'
DMID_DIR = f'{PROJECT}/data/external/dmid_mass_256_jit_v2/roi'
OUT      = f'{PROJECT}/results/p2_rev4'
ROOTS    = {'train': f'{PROJECT}/results/p2_jit_runs_train', 'val': f'{PROJECT}/results/p2_jit_runs_val',
            'train4': f'{PROJECT}/results/p2_rev4_runs_train', 'val4': f'{PROJECT}/results/p2_rev4_runs_val'}
SCHED    = {'train': 'train', 'val': 'val', 'train4': 'train', 'val4': 'val'}
for d in [OUT] + list(ROOTS.values()): os.makedirs(d, exist_ok=True)
PROTO_FILE = f'{OUT}/PROTOCOL_rev4_prespecified.txt'
PROTO = """Paper 2 - revision 4 analyses, written before any of them was run (frozen cohort selected, crops made or runs trained).

A. Frozen evaluation on untouched patients. The pipeline is frozen: the 190 saved models, crop protocol, threshold 0.5 and metrics are
   unchanged. Cohort: CBIS-DDSM mass ROIs (same eligibility as the study: every mass ROI whose full-field image is available, no filtering)
   of patients NOT among the 208 study patients. Patients are shuffled once with Python random.Random(2026) and added whole, in that
   order, while the total stays at or below 400 ROIs (patients that would exceed 400 are skipped); selection stops at 400 ROIs.
   Crops: the study's randomized protocol (seed from the MD5 of the ROI identifier). Every saved model predicts every frozen crop; per
   crop and seed, Dice is averaged over the five fold models (as for DMID). Primary estimands: the nine model contrasts of Tables VI-VII
   under each schedule, crossed hierarchical bootstrap over patients and seeds (10,000 replicates, rng(0)), Holm within the nine
   contrasts of each schedule. Secondary: model levels, the two no-network baselines, empty-prediction rates, HD95.
B. Inference checks (no training). (i) Exact inference repeated three times within one session (same checkpoint, same inputs) and
   compared with the stored values, for the 60 circuit runs and their Dense(4) partners, on the CBIS-DDSM test fold and DMID.
   (ii) Initialization audit: for seeds 0-2 and fold 0, the global and patch-wise models (circuit, Dense(4), trigonometric, identity)
   are built with the training seed; the weights of the layers shared by all variants are compared, and the initial block input and
   output statistics and the gradient norms at the block are recorded on four training batches.
C. New training runs (training and evaluation exactly as in the study):
   C. Trigonometric control g_tr, p_tr: z = tanh(A cos((pi/2) x + phi)), A 4x4 Glorot uniform, phi uniform [0, 2 pi): 20 parameters,
       same slot as the circuit; seeds 0-2, folds 0-4, both schedules (60 runs).
   Primary estimands (CBIS-DDSM test folds): circuit - trigonometric at each placement under each schedule (4 contrasts, Holm over
   the four). Secondary: trigonometric - Dense(4) and trigonometric - identity at each placement; the same contrasts on DMID
   (exploratory) and on the frozen cohort; code-replacement drops of the trigonometric models.
   No further seeds and no further learning-rate schedule are trained.
All runs and all results are reported; none is discarded."""
if not os.path.exists(PROTO_FILE):
    open(PROTO_FILE, 'w').write(PROTO + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
print('protocol:', open(PROTO_FILE).read().splitlines()[-1])
CODE_DIR = f'{PROJECT}/code'
if not os.path.exists(f'{CODE_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    CODE_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p in (CODE_DIR, f'{PROJECT}/code'):
    if p not in sys.path: sys.path.insert(0, p)


In [ ]:
# write the revision-4 module (trigonometric block) next to qseg2.py
open(f'{PROJECT}/code/qseg2_rev4.py', 'w').write('"""Revision-4 additions to qseg2 (Paper 2): a 20-parameter trigonometric block for the 4->4 slot.\n\n    z = tanh(A cos((pi/2) x + phi)),  A in R^{4x4} (Glorot uniform),  phi in R^4 (uniform on [0, 2 pi), as the circuit angles)\n\nThe angle encoding (pi/2) x is the one the circuit uses; like the circuit outputs, the block output is a trigonometric\nfunction of the angles. Global models apply it to the 4-value code, patch-wise models to every cell with one shared set of\nweights (as the patch-wise circuit). Models: g_tr, p_tr. Call patch() once after importing qseg2.\n"""\nimport numpy as np\nimport tensorflow as tf\nfrom tensorflow.keras import layers\n\nimport qseg2\n\n\nclass TrigLayer(layers.Layer):\n    def __init__(self, n=4, **kw):\n        super().__init__(**kw)\n        self.n = n\n\n    def build(self, input_shape):\n        self.A = self.add_weight(name="A", shape=(self.n, self.n), initializer="glorot_uniform")\n        self.phi = self.add_weight(name="phi", shape=(self.n,), initializer=tf.keras.initializers.RandomUniform(0.0, 2 * np.pi))\n        super().build(input_shape)\n\n    def call(self, x):\n        c = tf.cos(np.pi / 2 * tf.cast(x, tf.float32) + self.phi)\n        return tf.tanh(tf.einsum("...i,ji->...j", c, self.A))\n\n    def compute_output_shape(self, input_shape):\n        return input_shape\n\n    def get_config(self):\n        return {**super().get_config(), "n": self.n}\n\n\n_orig_block = qseg2._block\n\n\ndef _block(z, kind, n_qubits, n_layers, patch):\n    if kind == "tr":\n        return TrigLayer(n_qubits, name="trig_replacement")(z)\n    return _orig_block(z, kind, n_qubits, n_layers, patch)\n\n\ndef patch():\n    qseg2._block = _block\n    qseg2.MODELS2.setdefault("g_tr", "Global bottleneck, trigonometric 4->4 layer (20 parameters)")\n    qseg2.MODELS2.setdefault("p_tr", "Patch-wise bottleneck, trigonometric 4->4 layer (20 parameters)")\n    return qseg2\n')


In [ ]:
import importlib, tensorflow as tf, qseg2; importlib.reload(qseg2)
import qseg2_rev4; importlib.reload(qseg2_rev4); qseg2_rev4.patch()
from qseg.config import Config
cfg = Config(work_dir=OUT); B = cfg.batch_size
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
# 1 · Data + learning-rate rule switch ('train': halve on training loss; 'val': on validation loss)
cfg = Config(work_dir=OUT, max_epochs=200)
X, Y, DF = qseg2.load_cbis(CBIS_DIR)
print(X.shape, 'ROIs:', len(DF), '| patients:', DF.patient.nunique())
_Base = tf.keras.callbacks.ReduceLROnPlateau
while getattr(_Base, '_p2_switch', False): _Base = _Base.__mro__[1]
MONITOR = {'v': 'val_loss'}
class SwitchPlateau(_Base):
    _p2_switch = True
    def __init__(self, monitor='val_loss', **kw):
        super().__init__(monitor=('val_loss' if MONITOR['v'] == 'fixed' else MONITOR['v']), **kw)
        self.fixed = MONITOR['v'] == 'fixed'
    def on_epoch_end(self, epoch, logs=None):
        if self.fixed:
            logs = logs if logs is not None else {}; logs['learning_rate'] = float(self.model.optimizer.learning_rate.numpy()); return
        return super().on_epoch_end(epoch, logs)
tf.keras.callbacks.ReduceLROnPlateau = SwitchPlateau
def use_schedule(s):
    MONITOR['v'] = {'train': 'loss', 'val': 'val_loss', 'fixed': 'fixed'}[s]
for s in ('train', 'val'):
    use_schedule(s); cb = qseg2.tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4); print(s, '->', 'fixed' if cb.fixed else cb.monitor)

In [ ]:
# 2 · Weight files without optimizer state (as notebook 13, verified weight by weight)
def shrink_weights(run, root):
    d = f'{root}/{run}'; big = f'{d}/best.weights.h5'; small = f'{d}/best_small.weights.h5'
    if not os.path.exists(big) or os.path.exists(f'{d}/.shrunk'): return
    if os.path.getsize(big) < 60e6: open(f'{d}/.shrunk', 'w').close(); return
    name = run.split('_fold')[0]; tf.keras.backend.clear_session()
    m1 = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers); m1.load_weights(big); m1.save_weights(small)
    m2 = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers); m2.load_weights(small)
    assert all(np.array_equal(a, b) for a, b in zip(m1.get_weights(), m2.get_weights())), f'{run}: weight check failed'
    os.replace(small, big); open(f'{d}/.shrunk', 'w').close()

In [ ]:
# 3 · Training (resumable; sessions share the work through lock files)
SETS4 = [('C1', 'train4', [(m, f, s) for s in (0, 1, 2) for f in range(5) for m in ('g_tr', 'p_tr')]),
         ('C1', 'val4',   [(m, f, s) for s in (0, 1, 2) for f in range(5) for m in ('g_tr', 'p_tr')]),
]
print('runs planned:', sum(len(r) for _, _, r in SETS4))
LOCK_H = 4
def claim(root, run):
    d = f'{root}/{run}'
    if os.path.exists(f'{d}/summary.json'): return False
    lk = f'{d}/.lock'
    if os.path.exists(lk):
        who, t = open(lk).read().split()
        if who != SESSION and time.time() - float(t) < LOCK_H * 3600: return False
    os.makedirs(d, exist_ok=True); open(lk, 'w').write(f'{SESSION} {time.time()}'); time.sleep(20)
    return open(lk).read().split()[0] == SESSION
for part, key, runs in SETS4:
    root = ROOTS[key]; use_schedule(SCHED[key])
    cfg.to_json(f'{root}/train_config.json'); json.dump({'schedule': SCHED[key], 'part': part}, open(f'{root}/schedule.json', 'w'))
    order = runs if SESSION == 'A' else runs[::-1]
    for name, fold, seed in order:
        run = f'{name}_fold{fold}_seed{seed}'
        if not claim(root, run): continue
        s = qseg2.run_fold(cfg, (X, Y, DF), name, fold, seed, out_root=root, min_epochs=15, verbose=0)
        shrink_weights(run, root)
        n_done = sum(os.path.exists(f'{root}/{m}_fold{f}_seed{s_}/summary.json') for m, f, s_ in runs)
        print(time.strftime('%H:%M UTC', time.gmtime()), f'| {part}/{key} {run}: Dice {s["dice_per_image"]:.4f}, {s["epochs_run"]} epochs | {n_done}/{len(runs)}', flush=True)
print('session', SESSION, 'has nothing left to start')

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()